In [ ]:
import torch
from torchvision import datasets, transforms
from torchvision.models import vit_b_16,vit_b_32
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader,random_split
from sklearn.metrics import classification_report

NUM_CLASSES = 2
EPOCHS = 10
LEARNING_RATE = 1e-4


In [ ]:
import os
from torch.cuda import manual_seed
from google.colab import drive

drive.mount("/content/drive")

path = '/content/drive/MyDrive/Breast_Cancer/dataset_cancer_v1/classificacao_binaria/40X'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_datasets = datasets.ImageFolder(path, transform = transform)

train_size = int(0.8*len(full_datasets))
val_size = int(0.1*len(full_datasets))
test_size = len(full_datasets)- train_size - val_size

train_dataset,val_dataset,test_dataset = random_split(full_datasets,[train_size,val_size,test_size],generator=torch.Generator().manual_seed(42))


In [ ]:
#Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(x, dim=1, keepdim=True)

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x_cat = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        attention = self.sigmoid(
            self.conv(x_cat)
        )

        return x * attention



In [ ]:
#Window attention
class WindowAttention(nn.Module):

    def __init__(
        self,
        dim=768,
        num_heads=8
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        out, _ = self.attn(
            x,
            x,
            x
        )

        return out




In [ ]:
#Hybrid model
class HybridViT(nn.Module):

    def __init__(self, num_classes=2):

        super().__init__()

        self.vit16 = vit_b_16(weights="IMAGENET1K_V1")
        self.vit32 = vit_b_32(weights="IMAGENET1K_V1")

        feat16 = self.vit16.heads.head.in_features
        feat32 = self.vit32.heads.head.in_features

        self.vit16.heads = nn.Identity()
        self.vit32.heads = nn.Identity()

        self.fusion = nn.Linear(
            feat16 + feat32,
            768
        )

        self.spatial_attention = SpatialAttention()

        self.window_attention = WindowAttention(
            dim=768,
            num_heads=8
        )

        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        feat16 = self.vit16(x)
        feat32 = self.vit32(x)

        fused = torch.cat(
            [feat16, feat32],
            dim=1
        )

        fused = self.fusion(fused)

        B = fused.size(0)

        spatial_map = fused.view(
            B,
            768,
            1,
            1
        )

        spatial_map = self.spatial_attention(
            spatial_map
        )

        fused = spatial_map.view(
            B,
            1,
            768
        )

        fused = self.window_attention(
            fused
        )

        fused = fused.squeeze(1)

        output = self.classifier(
            fused
        )

        return output



In [ ]:
#MODEL

model = HybridViT(
    num_classes=NUM_CLASSES
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
import matplotlib.pyplot as plt

train_losses = []
val_losses = []

def train_one_epoch():

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            preds == labels
        ).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc


In [ ]:
def validate():

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += loss.item()

            _, preds = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                preds == labels
            ).sum().item()

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
best_val_acc = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc
        }, "best_hybrid_vit.pth")

print("\nTraining Complete")
print("Best Validation Accuracy:", best_val_acc)


In [ ]:
model.load_state_dict(
    torch.load(
        "best_hybrid_vit.pth",
        map_location=device
    )["model"]
)

model.eval()

all_preds = []
all_labels = []

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            preds == labels
        ).sum().item()

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

test_accuracy = 100 * correct / total

print("\nTest Accuracy:", test_accuracy)

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=full_datasets.classes
    )
)
